# Vocabulary & Token Shift Analysis
**Datasets:**
- A: `SinLlama-Dataset.txt` (10.7M lines, 3.0 GB) — the corpus SinLlama was already trained on
- B: `CPT-Dataset.txt` (8.7M lines, 2.4 GB) — the new continual-pretraining corpus

**Tokenizer:** **SinLlama** (`SinLlama_merged_bf16`) — Llama-3 byte-level BPE extended with Sinhala vocabulary (**139,336 tokens** = 128,256 Llama-3 base + 11,080 Sinhala). This is the tokenizer that will actually be used for continual pre-training, so the vocab/token shift is measured in *its* token space.

**Strategy:** Stream full datasets line-by-line — no RAM ceiling. All numbers are computed over the complete corpus.

---
## 1. Setup


In [ ]:
import collections
import warnings
from pathlib import Path
from tqdm import tqdm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from PIL import Image, ImageDraw, ImageFont
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from transformers import AutoTokenizer
from transformers import logging as hf_logging

warnings.filterwarnings("ignore")
hf_logging.set_verbosity_error()        # silence the "sequence longer than max length" notice
sns.set_theme(style="whitegrid", font_scale=1.2)
plt.rcParams["figure.dpi"] = 130

# Font for Sinhala labels — PIL+raqm does proper HarfBuzz shaping
# Swap this path to Iskola Pota .ttf if you have the file
SINHALA_FONT_PATH = "/usr/share/fonts/noto/NotoSerifSinhala-Regular.ttf"

def render_sinhala_label(text, font_size=13, color=(40, 40, 40)):
    """Render a Sinhala string to an RGBA numpy array via PIL + raqm shaping."""
    font = ImageFont.truetype(SINHALA_FONT_PATH, font_size,
                              layout_engine=ImageFont.Layout.RAQM)
    dummy = Image.new("RGBA", (1, 1))
    draw  = ImageDraw.Draw(dummy)
    bbox  = draw.textbbox((0, 0), text, font=font, language="si")
    w = max(bbox[2] - bbox[0] + 6, 1)
    h = max(bbox[3] - bbox[1] + 4, 1)
    img  = Image.new("RGBA", (w, h), (255, 255, 255, 0))
    draw = ImageDraw.Draw(img)
    draw.text((-bbox[0] + 3, -bbox[1] + 2), text, font=font,
              fill=(*color, 255), language="si")
    return np.array(img)

BASE      = Path("/ml/SinLlama_CPT/data_analysis")
FILE_A    = BASE / "SinLlama-Dataset.txt"
FILE_B    = BASE / "CPT-Dataset.txt"
MODEL_DIR = "/ml/SinLlama_CPT/SinLlama_merged_bf16"   # SinLlama tokenizer (Llama-3 BPE + Sinhala)
OUT_DIR   = BASE / "outputs"
OUT_DIR.mkdir(exist_ok=True)

# ── SinLlama tokenizer (byte-level BPE, vocab 139,336) ───────────────────────────
tok = AutoTokenizer.from_pretrained(MODEL_DIR)
VOCAB_SIZE = len(tok)                    # 139,336 incl. Sinhala + special tokens

# Byte-level BPE tokens aren't human-readable on their own (a Sinhala syllable is
# several UTF-8 bytes, each mapped to a placeholder char). convert_tokens_to_string
# reverses the byte-level encoding and returns proper text — with a leading space
# for word-initial tokens (the BPE equivalent of SentencePiece's ▁).
def id_to_text(i):
    return tok.convert_tokens_to_string([tok.convert_ids_to_tokens(int(i))])

print(f"Tokenizer  : SinLlama (Llama-3 byte-level BPE + Sinhala)")
print(f"Vocab size : {VOCAB_SIZE:,}")
print(f"Sinhala font: {Path(SINHALA_FONT_PATH).name}  |  raqm shaping: ON")

## 2. Full-corpus token frequency count
Streams each file, tokenises every line, accumulates counts. Runs in O(N) time with O(vocab) memory.

In [ ]:
def count_tokens(filepath, label, batch=4000):
    """Stream a file, batch-tokenise with the SinLlama tokenizer, accumulate ID counts.
    O(N) time, O(vocab) memory. add_special_tokens=False → no BOS/EOS in the counts."""
    counter = collections.Counter()
    total_lines = 0
    total_tokens = 0
    buf = []

    def flush():
        nonlocal total_lines, total_tokens
        enc = tok(buf, add_special_tokens=False, return_attention_mask=False)["input_ids"]
        for ids in enc:
            counter.update(ids)
            total_tokens += len(ids)
        total_lines += len(buf)
        buf.clear()

    with open(filepath, encoding="utf-8", errors="ignore") as f:
        for line in tqdm(f, desc=f"Tokenising {label}"):
            line = line.strip()
            if not line:
                continue
            buf.append(line)
            if len(buf) >= batch:
                flush()
        if buf:
            flush()

    print(f"{label}: {total_lines:,} lines | {total_tokens:,} tokens | {len(counter):,} unique token IDs")
    return counter, total_lines, total_tokens

counter_a, lines_a, tokens_a = count_tokens(FILE_A, "A (SinLlama)")
counter_b, lines_b, tokens_b = count_tokens(FILE_B, "B (CPT-Dataset)")


## 3. Vocabulary overlap & Jaccard similarity

In [ ]:
vocab_a = set(counter_a.keys())
vocab_b = set(counter_b.keys())

only_a      = vocab_a - vocab_b
only_b      = vocab_b - vocab_a
shared      = vocab_a & vocab_b
union       = vocab_a | vocab_b
jaccard     = len(shared) / len(union)

print(f"Unique to A:   {len(only_a):>8,}")
print(f"Unique to B:   {len(only_b):>8,}")
print(f"Shared:        {len(shared):>8,}")
print(f"Union:         {len(union):>8,}")
print(f"Jaccard:       {jaccard:.4f}")


### 3a. Vocabulary overlap bar chart

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(
    ["Only in A", "Only in B", "Shared"],
    [len(only_a), len(only_b), len(shared)],
    color=["#4C72B0", "#DD8452", "#55A868"],
    edgecolor="white", linewidth=0.8
)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f"{int(bar.get_height()):,}", ha="center", va="bottom", fontsize=11)
ax.set_ylabel("Number of token IDs")
ax.set_title(f"Vocabulary Overlap  |  Jaccard = {jaccard:.4f}")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.tight_layout()
plt.savefig(OUT_DIR / "01_vocab_overlap.png")
plt.show()
print("Saved → outputs/01_vocab_overlap.png")


## 4. Token frequency difference (top-K tokens)

In [ ]:
TOP_K = 40

# Normalise to relative frequency
freq_a = {k: v / tokens_a for k, v in counter_a.items()}
freq_b = {k: v / tokens_b for k, v in counter_b.items()}

all_tokens = union
diff = {t: freq_a.get(t, 0) - freq_b.get(t, 0) for t in all_tokens}
sorted_diff = sorted(diff.items(), key=lambda x: abs(x[1]), reverse=True)[:TOP_K]

tokens_sorted, diffs_sorted = zip(*sorted_diff)
# Decode each id to its readable surface form (byte-level BPE → text)
pieces = [id_to_text(t) for t in tokens_sorted]

colors = ["#4C72B0" if d > 0 else "#DD8452" for d in diffs_sorted]

fig, ax = plt.subplots(figsize=(13, 9))
ax.barh(range(TOP_K), diffs_sorted, color=colors, edgecolor="white")

# Hide y-tick text; we'll draw PIL-rendered images instead
ax.set_yticks(range(TOP_K))
ax.set_yticklabels([])
ax.tick_params(axis="y", length=0)

# Place each Sinhala label as a properly-shaped PIL image
for i, piece in enumerate(pieces):
    label_txt = piece if piece.strip() else repr(piece)   # show whitespace/byte-frag tokens
    img_arr = render_sinhala_label(label_txt, font_size=13)
    im = OffsetImage(img_arr, zoom=1.0)
    ab = AnnotationBbox(
        im, xy=(0, i),
        xycoords=("axes fraction", "data"),
        xybox=(-6, 0), boxcoords="offset points",
        box_alignment=(1.0, 0.5),
        frameon=False,
    )
    ax.add_artist(ab)

# Extra left margin so labels don't clip
fig.subplots_adjust(left=0.22)

ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Relative frequency  (A − B)")
ax.set_title(f"Top {TOP_K} tokens by |frequency difference|\n"
             f"(Blue = more in A · SinLlama, Orange = more in B · CPT-Dataset  |  leading space = word-initial token)")

from matplotlib.patches import Patch
ax.legend(handles=[Patch(color="#4C72B0", label="More in A (SinLlama)"),
                   Patch(color="#DD8452", label="More in B (CPT-Dataset)")], loc="lower right")

plt.savefig(OUT_DIR / "02_token_freq_diff.png", bbox_inches="tight")
plt.show()
print("Saved → outputs/02_token_freq_diff.png")

## 5. Token length distribution
Checks whether one corpus uses longer / shorter subword pieces on average.

In [ ]:
def piece_lengths(counter):
    """Character length of each token's decoded surface form, with its frequency.
    Computed once per *unique* token id (not per occurrence)."""
    lengths = []
    counts  = []
    for tid, cnt in counter.items():
        piece = id_to_text(tid)          # readable surface form (incl. any leading space)
        lengths.append(len(piece))
        counts.append(cnt)
    return np.array(lengths), np.array(counts)

len_a, cnt_a = piece_lengths(counter_a)
len_b, cnt_b = piece_lengths(counter_b)

# Weighted average piece length (weighted by how often each token actually occurs)
wavg_a = np.average(len_a, weights=cnt_a)
wavg_b = np.average(len_b, weights=cnt_b)
print(f"Weighted avg piece length  A: {wavg_a:.3f}  B: {wavg_b:.3f}")

fig, ax = plt.subplots(figsize=(8, 4))
max_len = int(max(len_a.max(), len_b.max())) + 1
bins = range(1, max_len + 1)

hist_a = np.bincount(len_a, weights=cnt_a, minlength=max_len)[1:]
hist_b = np.bincount(len_b, weights=cnt_b, minlength=max_len)[1:]
hist_a = hist_a / hist_a.sum()
hist_b = hist_b / hist_b.sum()
x = np.arange(1, max_len)

ax.bar(x - 0.2, hist_a, width=0.4, label=f"A — avg {wavg_a:.2f}", color="#4C72B0", alpha=0.85)
ax.bar(x + 0.2, hist_b, width=0.4, label=f"B — avg {wavg_b:.2f}", color="#DD8452", alpha=0.85)
ax.set_xlabel("Subword piece length (decoded chars)")
ax.set_ylabel("Proportion of tokens")
ax.set_title("Token (Subword Piece) Length Distribution")
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "03_piece_length_dist.png")
plt.show()
print("Saved → outputs/03_piece_length_dist.png")


## 6. Summary table

In [ ]:
summary = pd.DataFrame({
    "Metric": [
        "Total lines", "Total tokens", "Unique token IDs",
        "Tokens only in A", "Tokens only in B", "Shared tokens",
        "Jaccard similarity", "Avg piece length"
    ],
    "A (SinLlama)": [
        f"{lines_a:,}", f"{tokens_a:,}", f"{len(vocab_a):,}",
        f"{len(only_a):,}", "—", f"{len(shared):,}",
        f"{jaccard:.4f}", f"{wavg_a:.3f}"
    ],
    "B (CPT-Dataset)": [
        f"{lines_b:,}", f"{tokens_b:,}", f"{len(vocab_b):,}",
        "—", f"{len(only_b):,}", f"{len(shared):,}",
        f"{jaccard:.4f}", f"{wavg_b:.3f}"
    ]
})
summary


---

## 7. Notebook Summary

> **Roles:** **A = SinLlama-Dataset** — the corpus the SinLlama model was already trained on (existing knowledge). **B = CPT-Dataset** — the *new* corpus for continual pre-training.
> **Tokenizer:** measured in the **SinLlama token space** (Llama-3 byte-level BPE + Sinhala, 139,336 tokens) — the tokenizer that will actually be used for CPT.

### What we did

| Step | Description |
|------|-------------|
| **1. Setup** | Loaded the **SinLlama** tokenizer (`SinLlama_merged_bf16`, byte-level BPE, 139 K vocab). Registered a PIL + raqm (HarfBuzz) pipeline for proper Sinhala label rendering, plus a `id_to_text()` helper that decodes byte-level tokens back to readable text. |
| **2. Full-corpus token count** | Streamed both datasets entirely, batch-tokenised (no sampling). Counted 293 M tokens (A, SinLlama) and 219 M tokens (B, CPT). |
| **3. Vocabulary overlap** | Computed the sets of unique token IDs in each corpus, their intersection, union, and Jaccard similarity. |
| **4. Vocab overlap bar chart** | Visualised token IDs exclusive to A, exclusive to B, and shared (`01_vocab_overlap.png`). |
| **5. Token frequency difference** | Normalised counts to relative frequencies, computed `freq_A − freq_B`, plotted the top-40 tokens by absolute difference with decoded Sinhala labels (`02_token_freq_diff.png`). |
| **6. Piece length distribution** | Measured the decoded character length of every used token and plotted frequency-weighted histograms (`03_piece_length_dist.png`). |

---

### Key findings

#### Corpus size & token usage
| | A — SinLlama | B — CPT-Dataset |
|---|---|---|
| Lines | 10,730,154 | 8,696,658 |
| Total tokens | 293,228,756 | 219,475,655 |
| Tokens / line | **27.3** | **25.2** |
| Unique token IDs used | **78,324** | **12,719** |
| Vocab coverage (of 139,336) | **56.2 %** | **9.1 %** |

#### Vocabulary overlap
| Quantity | Value |
|---|---|
| Shared token IDs | 12,647 |
| Only in A (SinLlama) | **65,677** |
| Only in B (CPT) | **72** |
| Union | 78,396 |
| **Jaccard similarity** | **0.1613** |

- **B's token inventory is almost entirely a subset of A's:** 12,647 of B's 12,719 tokens (**99.4 %**) also occur in A; only **72** tokens are unique to B.
- The low Jaccard is driven by **asymmetry + corpus diversity, not a Sinhala lexical gap.** A is larger (293 M tokens) and far more lexically diverse — it exercises 65,677 tokens B never uses. Those are overwhelmingly the **Llama-3 base long tail** (Latin script, code, digits, punctuation, other languages, rare byte combos) that a big, mixed web corpus naturally hits, plus rarer Sinhala tokens.
- ⚠️ **Caveat:** a raw set-Jaccard conflates corpus *size/diversity* with genuine lexical shift, so **0.16 here is not comparable to HelaBERT's 0.9998** (that tokenizer had only a 32 K all-Sinhala vocab, so both corpora saturated it). The robust, decision-relevant fact is the **near-subset relationship B ⊂ A**.

#### Subword piece length
| | A — SinLlama | B — CPT-Dataset |
|---|---|---|
| Weighted avg piece length | **4.049 chars** | **4.337 chars** |

B's tokens are slightly **longer** on average — the CPT corpus is more purely Sinhala (longer whole-word Sinhala tokens), whereas A's average is pulled down by its heavier use of short base-vocab tokens (punctuation, single bytes, English subwords). This matches the HelaBERT result, where B also used the longer pieces.

The top frequency-difference tokens are **punctuation and function words** (`.`, `,`, `..`, ` බව`, ` ඇති`, ` අතර`, ` එක`, ZWNJ) — i.e. **stylistic/formatting** differences, not different content vocabulary.

---

### What this means for continual pre-training

- **No new tokens to learn.** B introduces only 72 tokens not already seen in A (all rare), so the model's existing embedding table already covers essentially everything the CPT corpus will present — **no tokenizer change or embedding resize, and no cold-start on unseen tokens.**
- **Lexically, the new data is "contained" within what the model already tokenizes** (B ⊂ A) — reassuring for CPT stability.
- **Mild long-tail forgetting risk:** A exercises ~65 K tokens (mostly base-Llama-3) that B won't reinforce. If training *only* on B, the model could drift on that tail — another argument for the light SinLlama (A) replay recommended in Notebook 2.
- **Lexical surface looks similar; the real differences are semantic/distributional** — which is exactly what **Notebook 2** measures (centroid distance, MMD, UMAP, clustering) and where the meaningful (though still mild) shift shows up.

---

## 8. Appendix — vocabulary overlap by script

The overall Jaccard in Section 3 is low (**0.16**), which seems to contradict the "no meaningful lexical shift" story. This appendix resolves it by splitting the used-token sets by **script** — Sinhala / Latin / Digit / Other.

**Finding:** on **Sinhala tokens alone** the two corpora are virtually identical (Jaccard ≈ **0.998**, matching the HelaBERT result). The entire overall gap is **non-Sinhala** content: A (the original web-scraped SinLlama corpus) uses tens of thousands of Latin/English tokens, while B (the cleaned CPT corpus) uses essentially none. So the low overall Jaccard reflects **corpus cleanliness / language purity**, not a Sinhala vocabulary shift — and it explains why HelaBERT (a 32 K all-Sinhala vocab with no Latin tokens) couldn't surface this difference at all.

In [ ]:
import re

sinh_re  = re.compile(r"[\u0d80-\u0dff]")   # Sinhala Unicode block (U+0D80–U+0DFF)
latin_re = re.compile(r"[A-Za-z]")
digit_re = re.compile(r"[0-9]")

def token_script(i):
    t = id_to_text(i)
    if sinh_re.search(t):  return "Sinhala"
    if latin_re.search(t): return "Latin"
    if digit_re.search(t): return "Digit"
    return "Other"                          # punctuation / symbols / whitespace / byte-frag

# Classify each *used* token id once (union of A and B)
script_of = {i: token_script(i) for i in union}

SCRIPTS = ["Sinhala", "Latin", "Digit", "Other"]
def compose(idset):
    c = collections.Counter(script_of[i] for i in idset)
    return [c.get(s, 0) for s in SCRIPTS]

groups = {
    "A (SinLlama)\nused": compose(vocab_a),
    "B (CPT)\nused":      compose(vocab_b),
    "Only in A":          compose(only_a),
    "Only in B":          compose(only_b),
    "Shared":             compose(shared),
}

# Sinhala-only overlap (the apples-to-apples comparison vs HelaBERT)
sinh_ids = {i for i in union if script_of[i] == "Sinhala"}
a_sinh, b_sinh = vocab_a & sinh_ids, vocab_b & sinh_ids
sinh_jaccard = len(a_sinh & b_sinh) / len(a_sinh | b_sinh)

print(f"Sinhala tokens used  — A: {len(a_sinh):,}   B: {len(b_sinh):,}   (of {len(sinh_ids):,})")
print(f"Sinhala-only Jaccard : {sinh_jaccard:.4f}   (overall Jaccard = {jaccard:.4f})")
print()
for g, vals in groups.items():
    print(f"  {g.replace(chr(10),' '):18s} " + "  ".join(f"{s}:{v:,}" for s, v in zip(SCRIPTS, vals)))

# ── stacked bar chart ────────────────────────────────────────────────────────────
colors = {"Sinhala": "#55A868", "Latin": "#C44E52", "Digit": "#8172B3", "Other": "#CCB974"}
labels = list(groups.keys())
data   = np.array([groups[l] for l in labels])      # rows = groups, cols = scripts

fig, ax = plt.subplots(figsize=(10, 5))
bottom = np.zeros(len(labels))
for j, s in enumerate(SCRIPTS):
    ax.bar(labels, data[:, j], bottom=bottom, label=s, color=colors[s], edgecolor="white", linewidth=0.6)
    bottom += data[:, j]

ax.set_ylabel("Number of token IDs")
ax.set_title("Vocabulary Overlap by Script — SinLlama tokenizer\n"
             f"Sinhala-only Jaccard = {sinh_jaccard:.4f}   vs   overall Jaccard = {jaccard:.4f}")
ax.legend(title="Script")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.tight_layout()
plt.savefig(OUT_DIR / "04_vocab_overlap_by_script.png", bbox_inches="tight")
plt.show()
print("\nSaved → outputs/04_vocab_overlap_by_script.png")